In [4]:
import sys
import os
import pandas as pd

from anthropic import Anthropic
from dotenv import load_dotenv

In [3]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [9]:
df_ground_truth = pd.read_csv('data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [10]:
ground_truth[:5]

[{'question': 'Is it too late to start this course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to join before a certain date to get a certificate?',
  'document': '74eb249bbf'},
 {'question': "Can I still enroll and complete the course if I'm joining late?",
  'document': '74eb249bbf'},
 {'question': "What's the deadline for submitting my project to earn the certificate?",
  'document': '74eb249bbf'},
 {'question': 'If I start now, will I be able to get certified for this course?',
  'document': '74eb249bbf'}]

In [11]:
from utils.ingest import load_faq_data, build_index

In [12]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [13]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [25]:
from toyaikit.llm import OpenAIClient, AnthropicClient

In [16]:
load_dotenv()

True

In [26]:
openai_client = OpenAIClient()

In [17]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [32]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner, AnthropicMessagesRunner

In [19]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [20]:
instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

In [33]:
runner = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=AnthropicClient(model="claude-haiku-4-5")
)

In [34]:
rec = ground_truth[0]

In [35]:
result = runner.loop(prompt=rec["question"])

In [44]:
result.all_messages

[{'role': 'system',
  'content': "You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering."},
 {'role': 'user',
  'content': 'Is it too late to start this course if I just found out about it?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll search the FAQ for information about starting the course late.", type='text'),
   ToolUseBlock(id='toolu_01JDdY1vQY94SDgnc27cXGYN', caller=DirectCaller(type='direct'), input={'query': 'late start join course after it has begun'}, name='search', type='tool_use')]},
 {'role': 'tool',
  'tool_call_id': 'toolu_01JDdY1vQY94SDgnc27cXGYN',
  'content': '[\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also jus

In [115]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if not isinstance(message['content'], list):
            continue

        if message['content'][-1].type == "tool_use":
            tool_calls.append({
                "name": message['content'][-1].name,
                "arguments": message['content'][-1].input,
            })

    return tool_calls

In [116]:
tool_calls = extract_tool_calls(result.all_messages)

In [117]:
tool_calls

[{'name': 'search',
  'arguments': {'query': 'late start join course after it has begun'}}]

In [118]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [119]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

In [120]:
agent_result

{'question': 'Is it too late to start this course if I just found out about it?',
 'answer_agent': "Good news! It's not too late to join! According to the FAQ:\n\n**Yes, you can still join the course** even if you just found out about it. \n\nHere are the key points:\n\n1. **You can start whenever you want** - All the videos and GitHub materials are available now, so you can begin learning immediately.\n\n2. **However, there's an important caveat for certificates**: If you want to receive a certificate, you'll need to submit your capstone project while submissions are still being accepted. The certificate requires completing the full course with a live cohort, which includes peer-reviewing 3 capstone projects.\n\n3. **Check the deadlines** - You can find the homework and submission deadlines in the [course management platform](https://courses.datatalks.club/llm-zoomcamp/).\n\nSo I'd recommend:\n- Start the course materials right away since they're all available\n- Keep an eye on the su

In [121]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [122]:
from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [123]:
df_agent = pd.DataFrame(agent_answers)

In [124]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Is it too late to start this course if I just ...,Good news! It's **not too late** to start this...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': {'query': 'la...",0.003812,74eb249bbf
1,Do I need to join before a certain date to get...,"Based on the FAQ results, here's what you need...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': {'query': 'ce...",0.00338,74eb249bbf
2,Can I still enroll and complete the course if ...,"Based on the FAQ results, here's what you need...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': {'query': 'la...",0.003972,74eb249bbf
3,What's the deadline for submitting my project ...,"Based on the FAQ results, I can see informatio...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': {'query': 'pr...",0.003669,74eb249bbf
4,"If I start now, will I be able to get certifie...","Based on the FAQ, **whether you can get certif...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': {'query': 'co...",0.005178,74eb249bbf


In [125]:
df_agent['cost'].sum()

Decimal('0.201663')

In [126]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [127]:
agent_answers = df_agent.to_dict(orient='records')

In [129]:
agent_answers[0]

{'question': 'Is it too late to start this course if I just found out about it?',
 'answer_agent': 'Good news! It\'s **not too late** to start this course. According to the FAQ:\n\n**You can start whenever you want.** The videos and course materials on GitHub are available, so you can begin learning immediately regardless of when you find out about the course.\n\nHere\'s the typical workflow:\n1. Watch the lesson videos\n2. Work through the lesson notebooks/code\n3. Read the homework instructions on GitHub\n4. Submit answers through the course platform before the deadline\n\nA few things to keep in mind:\n- **Deadlines still apply** - While you can start anytime, homework submissions do have deadlines listed in the course management platform\n- **Self-paced vs. Live cohort** - If you\'re interested in getting a certificate, you\'ll need to complete the course with a "live" cohort (not in pure self-paced mode), since certificates require peer-reviewing 3 capstone projects\n- **No regist

In [130]:
from pydantic import BaseModel, Field
from typing import Literal

In [131]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [132]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

In [133]:
agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [134]:
import json
from utils.evaluation_utils import calc_total_price, llm_structured_retry

In [139]:
anthropic_client = Anthropic()

In [137]:
def evaluate_agent_answer(rec, model="claude-haiku-4-5"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        anthropic_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [140]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The original answer states that yes, it is too late if you want to receive a certificate, but you need to submit your project while submissions are still being accepted. The agent's answer contradicts the original answer by saying 'It's NOT too late to start this course' and provides different information about certificates requiring peer-reviewing capstone projects and joining a live cohort. The agent's answer does not align with the ground truth which affirms that it IS too late (with a caveat about submission deadlines). The agent fundamentally misrepresents the answer to the question.", answer_score='bad', trajectory_reasoning="The tool call used the query 'late enrollment joining course after start' which includes relevant keywords like 'late,' 'enrollment,' and 'course.' This is a reasonable search query for the question asked. However, only one search was performed, and based on the mismatch between the agent's answer and the ground truth, it ap

In [141]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [142]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [143]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [144]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [145]:
calc_total_price(usages)

0.11322499999999999

In [146]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    43
bad      7
Name: count, dtype: int64

In [147]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    45
bad      5
Name: count, dtype: int64

In [148]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)